In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "refinement-per-claim"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

### Users' data are not shared in this repo as mentioned in the paper. Output of cells in this notebook have been cleared because they reveal info about accounts

In [ ]:
import pandas as pd

df = pd.read_excel("../Data/pro_russian_users_data_for_agents.xlsx")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(2)

In [ ]:
import ast

# Each row is one user; 'text' column holds a list of tweets as a string
df['tweet_list'] = df['text'].apply(ast.literal_eval)
df['tweet_count'] = df['tweet_list'].apply(len)

tweets_per_user = df[['author_username', 'tweet_count']].sort_values('tweet_count', ascending=False)
print("Tweets per user:")
print(tweets_per_user.to_string(index=False))
print(f"\nTotal tweets across all users: {df['tweet_count'].sum()}")

In [ ]:
# --- Deterministic data splitting (no leakage) ---
import random
random.seed(42)
splits = {}  # keyed by author_username

for _, row in df.iterrows():
    uname = row['author_username']
    all_pos = ast.literal_eval(row['text'])

    # Pool all tweets from every other user
    all_neg = []
    for _, other_row in df[df['author_username'] != uname].iterrows():
        all_neg.extend(ast.literal_eval(other_row['text']))

    # Sample for the agent prompt, leaving at least 1 positive tweet for the test set
    n_pos_prompt = min(10, max(0, len(all_pos) - 1))
    prompt_pos = random.sample(all_pos, n_pos_prompt)
    n_neg_prompt = min(10, max(0, len(all_neg) - 5))
    prompt_neg = random.sample(all_neg, n_neg_prompt)

    # Test set: only tweets NOT included in the prompt
    remaining_pos = [t for t in all_pos if t not in prompt_pos]
    remaining_neg = [t for t in all_neg if t not in prompt_neg]
    test_pos = random.sample(remaining_pos, min(5, len(remaining_pos)))
    test_neg = random.sample(remaining_neg, min(5, len(remaining_neg)))

    splits[uname] = {
        'prompt_pos': prompt_pos,
        'prompt_neg': prompt_neg,
        'test_pos':   test_pos,
        'test_neg':   test_neg,
        'summary':    row['detailed_summary'],
    }

# Sanity check — no tweet can appear in both prompt and test
for uname, s in splits.items():
    assert not set(s['test_pos']) & set(s['prompt_pos']), f"Positive leakage for {uname}"
    assert not set(s['test_neg']) & set(s['prompt_neg']), f"Negative leakage for {uname}"

print("Split complete. No leakage detected.\n")
print(f"{'User':<22} {'prompt_pos':>10} {'prompt_neg':>10} {'test_pos':>8} {'test_neg':>8}")
print("-" * 60)
for uname, s in splits.items():
    print(f"{uname:<22} {len(s['prompt_pos']):>10} {len(s['prompt_neg']):>10} {len(s['test_pos']):>8} {len(s['test_neg']):>8}")

In [ ]:
# --- Classification system prompt builder ---
def build_classification_prompt(uname, summary, prompt_pos, prompt_neg):
    pos_str = "\n".join([f"{i+1}. {t}" for i, t in enumerate(prompt_pos)])
    neg_str = "\n".join([f"{i+1}. {t}" for i, t in enumerate(prompt_neg)])
    return f"""You are {uname}, a strongly pro-Russian user on X.
Below is a detailed summary of your tweet history, capturing key themes, tone, and overall perspective:
-------------------------------
{summary}
-------------------------------

Here are {len(prompt_pos)} of your own tweets. Study them carefully — they are ground truth examples of how YOU write:
-------------------------------
{pos_str}
-------------------------------

Here are {len(prompt_neg)} tweets written by OTHER pro-Russian users (not you). They share your political stance but each has a distinct voice, style, and narrative focus:
-------------------------------
{neg_str}
-------------------------------

Your task: You will be given a single tweet. Decide whether YOU wrote it.

To make an accurate decision, use the examples above as follows:

- Use your own tweets to identify what is genuinely distinctive about your writing: your formatting habits, sentence structure, recurring phrases, tonal register, and the specific angles or sub-topics you tend to focus on.
- Use the other users' tweets to calibrate what is truly distinctive to you versus what is broadly shared among pro-Russian accounts. A feature that appears across many of the other users' tweets is not a reliable marker of your authorship.
- Apply both signals together: a tweet is likely yours if it matches your distinctive patterns and does not fit the style of the other users. A tweet is likely not yours if it lacks your distinctive patterns or resembles the style of the other users.
- Be flexible: not every tweet you write will contain all of your recurring markers. Consider whether the overall voice and structure are consistent with how you write, rather than requiring every characteristic to be present.

Output ONLY a valid JSON object with this exact structure:
{{
  "is_mine": true or false,
  "reasoning": "2-3 sentences identifying the specific stylistic or thematic features that drove your decision, with reference to how the other users' examples informed your judgment"
}}
No extra text. No markdown. Only the JSON object.
"""

# Spot-check one prompt
sample_user = list(splits.keys())[0]
s = splits[sample_user]
sample_prompt = build_classification_prompt(sample_user, s['summary'], s['prompt_pos'], s['prompt_neg'])
print(f"=== Sample prompt for '{sample_user}' ===")
print(sample_prompt, "...")

In [ ]:
%pip install smolagents litellm -q

In [ ]:
# --- Create classification agents ---
import os, importlib.resources, yaml
from smolagents import ToolCallingAgent, LiteLLMModel, PromptTemplates

# Set your Gemini API key here (or set it as an environment variable before running)
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

MODEL = "gemini/gemini-2.5-flash"

# Load default smolagents prompt templates, then override only system_prompt
def make_prompt_templates(sys_prompt):
    defaults = yaml.safe_load(
        importlib.resources.files("smolagents.prompts")
        .joinpath("toolcalling_agent.yaml")
        .read_text()
    )
    defaults['system_prompt'] = sys_prompt
    return defaults

classification_agents = {}
for uname, s in splits.items():
    sys_prompt = build_classification_prompt(
        uname, s['summary'], s['prompt_pos'], s['prompt_neg']
    )
    model = LiteLLMModel(
        MODEL,
        temperature=1,       # required by Gemini when thinking is enabled
        model_kwargs={
            "thinking": {"type": "enabled", "budget_tokens": 5000}
        },
    )
    agent = ToolCallingAgent(
        tools=[],
        model=model,
        verbosity_level=0,
        prompt_templates=make_prompt_templates(sys_prompt),
        add_base_tools=False,
        name=f"ClassificationAgent_{uname}",
    )
    classification_agents[uname] = agent

print(f"Created {len(classification_agents)} classification agents.")

In [ ]:
# --- Cell 7: Run classification loop ---
import json, time, re

RESULTS_CSV = "../Data/impersonation_validation_results_v2.csv"
results = []

for uname, agent in classification_agents.items():
    test_tweets = (
        [(t, True)  for t in splits[uname]['test_pos']] +
        [(t, False) for t in splits[uname]['test_neg']]
    )
    random.shuffle(test_tweets)
    print(f"Running {uname} ({len(test_tweets)} tweets)...", end=" ", flush=True)

    for tweet_text, ground_truth in test_tweets:
        query = f'Tweet: "{tweet_text}"'
        try:
            raw = agent.run(query, reset=True)
            match = re.search(r'\{.*?\}', str(raw), re.DOTALL)
            parsed = json.loads(match.group()) if match else {}
            predicted = parsed.get('is_mine', None)
            reasoning = parsed.get('reasoning', '')
        except Exception as e:
            raw, predicted, reasoning = str(e), None, ''

        results.append({
            'author':       uname,
            'tweet':        tweet_text,
            'label':        ground_truth,
            'predicted':    predicted,
            'reasoning':    reasoning,
            'raw_response': str(raw),
        })
        time.sleep(0.5)  # light rate-limiting

    print("done.")

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f"\nSaved {len(results_df)} classifications to {RESULTS_CSV}")
results_df.head()

In [ ]:
# --- Cell 7b: Retry dropped rows with more robust JSON parsing ---
def robust_parse(raw_str):
    """Try progressively looser strategies to extract is_mine from a raw response."""
    # Strategy 1: greedy JSON block (handles reasoning with embedded quotes/newlines)
    match = re.search(r'\{.*\}', raw_str, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            return parsed.get('is_mine', None), parsed.get('reasoning', '')
        except json.JSONDecodeError:
            pass

    # Strategy 2: just pull the boolean directly — avoids broken string fields
    match = re.search(r'"is_mine"\s*:\s*(true|false)', raw_str, re.IGNORECASE)
    if match:
        return match.group(1).lower() == 'true', ''

    return None, ''

dropped_idx = results_df[results_df['predicted'].isna()].index
print(f"Re-running {len(dropped_idx)} dropped rows...")

for idx in dropped_idx:
    row = results_df.loc[idx]
    uname = row['author']
    tweet_text = row['tweet']
    agent = classification_agents[uname]

    query = f'Tweet: "{tweet_text}"'
    try:
        raw = agent.run(query, reset=True)
        predicted, reasoning = robust_parse(str(raw))
    except Exception as e:
        raw, predicted, reasoning = str(e), None, ''

    results_df.at[idx, 'raw_response'] = str(raw)
    results_df.at[idx, 'predicted'] = predicted
    results_df.at[idx, 'reasoning'] = reasoning
    status = f"is_mine={predicted}" if predicted is not None else "STILL FAILED"
    print(f"  {uname}: {status}")

results_df.to_csv(RESULTS_CSV, index=False)
still_dropped = results_df['predicted'].isna().sum()
print(f"\nDone. Remaining unparseable: {still_dropped}")

In [ ]:
# --- Cell 8: Evaluation & summary ---
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── Change this to the CSV you want to evaluate ──────────────────────────────
EVAL_CSV = "../Data/impersonation_validation_results_v2.csv"
# EVAL_CSV = "../Data/impersonation_validation_results.csv"   # v1 (original)
# ─────────────────────────────────────────────────────────────────────────────

results_df = pd.read_csv(EVAL_CSV)
valid = results_df.dropna(subset=['predicted'])
n_dropped = len(results_df) - len(valid)
print(f"Loaded {len(results_df)} rows from '{EVAL_CSV}' ({n_dropped} dropped)\n")

print("=== Per-Agent Results ===\n")
print(f"{'User':<22} {'n':>4}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
print("-" * 58)

for uname in valid['author'].unique():
    sub = valid[valid['author'] == uname]
    y_true = sub['label'].tolist()
    y_pred = sub['predicted'].tolist()
    print(
        f"{uname:<22} {len(sub):>4}  "
        f"{accuracy_score(y_true, y_pred):>6.2f}  "
        f"{precision_score(y_true, y_pred, zero_division=0):>6.2f}  "
        f"{recall_score(y_true, y_pred, zero_division=0):>6.2f}  "
        f"{f1_score(y_true, y_pred, zero_division=0):>6.2f}"
    )

print("-" * 58)
y_true_all = valid['label'].tolist()
y_pred_all = valid['predicted'].tolist()
print(
    f"{'OVERALL':<22} {len(valid):>4}  "
    f"{accuracy_score(y_true_all, y_pred_all):>6.2f}  "
    f"{precision_score(y_true_all, y_pred_all, zero_division=0):>6.2f}  "
    f"{recall_score(y_true_all, y_pred_all, zero_division=0):>6.2f}  "
    f"{f1_score(y_true_all, y_pred_all, zero_division=0):>6.2f}"
)
if n_dropped:
    print(f"\n({n_dropped} responses dropped — unparseable JSON)")